In [3]:
# making somewhat cleaned-up versions of figures for figure 2 + related supplementary figures

In [4]:
import utils # ./utils.py
import os
import re
import math
import random
import pandas as pd
import numpy as np
import scipy as sp
from scipy import stats
import itertools
from itertools import combinations
import seaborn as sns
import matplotlib 
import matplotlib.pyplot as plt
from matplotlib import gridspec
from matplotlib import cm
import matplotlib.font_manager
%matplotlib inline

In [ ]:
# pre-processed data:
# excluded samples: SK1_8 (very low coverage), SK1_11 (duplicate of SK1_10), LS01_17 (no spike-in detected), LS01_32 (no reads)
metadata = pd.read_csv('/Users/alyulina/Projects/Cancer/Data/metadata.csv', index_col='sample id') # all sequencing samples
read_counts_by_cell_line = pd.read_csv('/Users/alyulina/Projects/Cancer/Data/cell-line_read_counts.csv', index_col='clID')
cell_counts_by_cell_line = pd.read_csv('/Users/alyulina/Projects/Cancer/Data/cell-line_cell_counts.csv', index_col='clID')

clIDs = utils.clIDs
clID__label = utils.clID__label
clID__color = utils.clID__color 

# where to find data:
path_to_reads = '/Users/alyulina/Projects/Cancer/Data/bc_counts/'

# where to save plots:
path_to_save_figs = '/Users/alyulina/Projects/Cancer/Figures/fig2/'

In [ ]:
# note that the figures might change slightly each time you run this code, 
# so make sure to update annotations accordingly

### In vitro plots

In [ ]:
in_vitro_samples = metadata.loc[metadata['experiment'] == 'in vitro'].groupby('time point, d').apply(lambda df: df.index.tolist()) 
clIDs_ordered_in_vitro = ['AGGT', 'AGCA', 'CTTC', 'AGTC', 'ATCG', 'GAAG', 'CACT', 'ACCT', 'CCTT', 'CATG', 'GGAA', 'ACAC', 'GTTG', 'AGAG', 'TTGG', 'TCCA', 'CTGT', 'TTCC', 'CAGA', 'GCAT', 'CGTA', 'GGTT', 'ACGA', 'AAGG', 'GTAC', 'CGAT', 'GCTA', 'CAAC', 'TGAC', 'ATGC', 'ACTG', 'GTGA', 'CCAA'] # clIDs sorted by weighted avg expansion in vitro

In [ ]:
plt.figure(figsize=(4, 4));
plt.gca().spines[['top', 'right']].set_visible(False)
plt.gca().spines['bottom'].set_position(('axes', 0))
plt.gca().spines['left'].set_position(('axes', 0))
w = 0.5


cmap = matplotlib.colors.LinearSegmentedColormap.from_list('custom_cmap', ['#c13e2e', '#e86b5a', '#f9a28d', '#e9e0c8', '#a3c3c3', '#5e9faf', '#00768d'][::-1])

colors = matplotlib.cm.ScalarMappable(norm=matplotlib.colors.TwoSlopeNorm(vmin=-2, vcenter=0, vmax=1), 
                                      cmap=cmap)

# defining cell line colors here
clID__color = {}
ls = np.logspace(1.1, -2.2, len(clIDs_ordered_in_vitro))
for count, clID in enumerate(clIDs_ordered_in_vitro):  
    
    xs = []
    ys = []
    errs = []
    
    # normalizing by the avg fraction at t = 3 days
    t = 3
    norm = np.mean([read_counts_by_cell_line[sample_id][clID] / sum(read_counts_by_cell_line[sample_id][1:]) for sample_id in in_vitro_samples[t]])
    
    for time_point, sample_ids in in_vitro_samples.items():
        
        xs.append(time_point)
        
        fracs = [read_counts_by_cell_line[sample_id][clID] / sum(read_counts_by_cell_line[sample_id][1:]) for sample_id in sample_ids]
        ys.append(np.mean(fracs / norm))
        errs.append(np.std(fracs / norm))
            
    plt.errorbar(xs, ys, errs, color=colors.to_rgba(np.log10(ys[-1])), linewidth=2*w, alpha=1, clip_on=False)
    clID__color[clID] = colors.to_rgba(np.log10(ys[-1]))
   
    plt.text(26, ls[count], clID__label[clID], size=8, va='center')
    plt.plot([xs[-1], 25.5], [ys[-1], ls[count]], linewidth=1.25*w, linestyle=':', color='black', clip_on=False, zorder=-1)
    
    plt.scatter(25.5, ls[count], color=colors.to_rgba(np.log10(ys[-1])), edgecolor='black', linewidth=w, s=20, clip_on=False)
    
plt.plot([t, xs[-1]], [1, 1], color='black', zorder=100, linewidth=w)

plt.annotate('', xy=(28.8, ls[2]), xycoords='data', xytext=(28.8, ls[3]), textcoords='data',
             arrowprops=dict(arrowstyle='-', color='0', patchA=None, patchB=None, connectionstyle='bar,fraction=0.7'), 
             annotation_clip=False);

plt.annotate('', xy=(28.8, ls[4]), xycoords='data', xytext=(28.8, ls[6]), textcoords='data',
             arrowprops=dict(arrowstyle='-', color='0', patchA=None, patchB=None, connectionstyle='bar,fraction=0.4'), 
             annotation_clip=False);

plt.annotate('', xy=(28.8, ls[18]), xycoords='data', xytext=(28.8, ls[19]), textcoords='data',
             arrowprops=dict(arrowstyle='-', color='0', patchA=None, patchB=None, connectionstyle='bar,fraction=0.7'), 
             annotation_clip=False);

plt.annotate('', xy=(28.8, ls[22]), xycoords='data', xytext=(28.8, ls[23]), textcoords='data',
             arrowprops=dict(arrowstyle='-', linestyle=':', color='0', patchA=None, patchB=None, connectionstyle='bar,fraction=0.7'), 
             annotation_clip=False, rotation=90);
   
plt.yscale('log'); plt.minorticks_off();
plt.xlim(t, 24)
plt.ylim(1e-2, 1e1)
plt.xticks([3, 6, 9, 12, 15, 18, 21, 24], size=12)
plt.yticks([1e-2, 1e-1, 1e0, 1e1], ['0.01', '0.1', '1', '10'], size=12)

plt.xlabel('Time point, days', fontsize=14, labelpad=8);
plt.ylabel('Relative ' + '$\it{in\,vitro}$' + ' expansion', fontsize=14, labelpad=8);

#plt.savefig(path_to_save_figs + '2d_in_vitro_time_series_w_labels.pdf', dpi=300, bbox_inches='tight')    


In [ ]:
plt.figure(figsize=(4, 4));
plt.gca().spines[['top', 'right']].set_visible(False)
plt.gca().spines['bottom'].set_position(('axes', 0))
plt.gca().spines['left'].set_position(('axes', 0))
w = 0.5


ls = np.logspace(1.1, -2.2, len(clIDs_ordered_in_vitro))
for count, clID in enumerate(clIDs_ordered_in_vitro): 
    if clID in ['CTTC', 'AGTC', 'ATCG', 'CACT', 'CAGA', 'GCAT', 'ACGA', 'AAGG']:
        color = clID__color[clID]
        z=1
        a=4
    else:
        color = '#e9e9e9'
        z=-1
        a=2
    
    xs = []
    ys = []
    errs = []
    
    # normalizing by the avg fraction at t = 3 days
    t = 3
    norm = np.mean([read_counts_by_cell_line[sample_id][clID] / sum(read_counts_by_cell_line[sample_id][1:]) for sample_id in in_vitro_samples[t]])
    
    for time_point, sample_ids in in_vitro_samples.items():
        
        xs.append(time_point)
        
        fracs = [read_counts_by_cell_line[sample_id][clID] / sum(read_counts_by_cell_line[sample_id][1:]) for sample_id in sample_ids]
        ys.append(np.mean(fracs / norm))
        errs.append(np.std(fracs / norm))
            
    plt.errorbar(xs, ys, errs, color=color, zorder=z, linewidth=a*w, alpha=1, clip_on=False)

    if clID in ['CTTC', 'AGTC', 'ATCG', 'CACT', 'CAGA', 'GCAT', 'ACGA', 'AAGG']:
        plt.text(26, ls[count], clID__label[clID], size=8, va='center')
        plt.plot([xs[-1], 25.5], [ys[-1], ls[count]], linewidth=1.25*w, linestyle=':', color='black', clip_on=False, zorder=-1)
        plt.scatter(25.5, ls[count], color=color, edgecolor='black', linewidth=w, s=20, clip_on=False)
    
plt.plot([t, xs[-1]], [1, 1], color='black', zorder=100, linewidth=w)

plt.annotate('', xy=(28.8, ls[2]), xycoords='data', xytext=(28.8, ls[3]), textcoords='data',
             arrowprops=dict(arrowstyle='-', color='0', patchA=None, patchB=None, connectionstyle='bar,fraction=0.7'), 
             annotation_clip=False);

plt.annotate('', xy=(28.8, ls[4]), xycoords='data', xytext=(28.8, ls[6]), textcoords='data',
             arrowprops=dict(arrowstyle='-', color='0', patchA=None, patchB=None, connectionstyle='bar,fraction=0.4'), 
             annotation_clip=False);

plt.annotate('', xy=(28.8, ls[18]), xycoords='data', xytext=(28.8, ls[19]), textcoords='data',
             arrowprops=dict(arrowstyle='-', color='0', patchA=None, patchB=None, connectionstyle='bar,fraction=0.7'), 
             annotation_clip=False);

plt.annotate('', xy=(28.8, ls[22]), xycoords='data', xytext=(28.8, ls[23]), textcoords='data',
             arrowprops=dict(arrowstyle='-', linestyle=':', color='0', patchA=None, patchB=None, connectionstyle='bar,fraction=0.7'), 
             annotation_clip=False, rotation=90);

   
plt.yscale('log'); plt.minorticks_off();
plt.xlim(t, 24)
plt.ylim(1e-2, 1e1)
plt.xticks([3, 6, 9, 12, 15, 18, 21, 24], size=12)
plt.yticks([1e-2, 1e-1, 1e0, 1e1], ['0.01', '0.1', '1', '10'], size=12)

plt.xlabel('Time point, days', fontsize=14, labelpad=8);
plt.ylabel('Relative ' + '$\it{in\,vitro}$' + ' expansion', fontsize=14, labelpad=8);

#plt.savefig(path_to_save_figs + '2si_in_vitro_time_series_w_labels_reps.pdf', dpi=300, bbox_inches='tight')    


In [ ]:
# looking at correlations between pairs of frequency trajectories

plt.figure(figsize=(4, 4));

r2 = {}
reps = ['CTTC', 'AGTC', 'ATCG', 'CACT', 'CAGA', 'GCAT', 'ACGA', 'AAGG']

for (x, y) in itertools.combinations(clIDs_ordered_in_vitro, 2):
        
    xs = []
    ys = []
    
    # normalizing by the avg fraction at t = 3 days
    t = 3
    norm_1 = np.mean([read_counts_by_cell_line[sample_id][x] / sum(read_counts_by_cell_line[sample_id][1:]) for sample_id in in_vitro_samples[t]])
    norm_2 = np.mean([read_counts_by_cell_line[sample_id][y] / sum(read_counts_by_cell_line[sample_id][1:]) for sample_id in in_vitro_samples[t]])

    for time_point, sample_ids in in_vitro_samples.items():
        
        fracs_1 = [read_counts_by_cell_line[sample_id][x] / sum(read_counts_by_cell_line[sample_id][1:]) for sample_id in sample_ids]
        xs.append(np.mean(fracs_1 / norm_1))
        
        fracs_2 = [read_counts_by_cell_line[sample_id][y] / sum(read_counts_by_cell_line[sample_id][1:]) for sample_id in sample_ids]
        ys.append(np.mean(fracs_2 / norm_2))
    
        
    r2[(x, y)] = stats.pearsonr(xs, ys)  
        

        
plt.hist([x[0] for x in r2.values()], bins=100, color='#b9b9b9')

for rep in [('CTTC', 'AGTC'), ('ATCG', 'CACT'), ('CAGA', 'GCAT'), ('ACGA', 'AAGG')]:
    plt.axvline(r2[rep][0], color='black')

plt.xlim(-1, 1)
plt.ylim(0, 100)

plt.xticks(size=12)
plt.yticks(size=12)


ax = plt.gca()
axins = ax.inset_axes([0.25, 0.25, 0.5, 0.5])
axins.hist([x[0] for x in r2.values()], bins=100, color='#b9b9b9');

for count, rep in enumerate([('CTTC', 'AGTC'), ('ATCG', 'CACT'), ('CAGA', 'GCAT'), ('ACGA', 'AAGG')]):
    axins.axvline(r2[rep][0], color='black', linewidth=2*w)
    # print(clID__label[rep[0]], r2[rep][0])
    
    if count == 1:
        axins.text(0.95, 112, clID__label[rep[0]], ha='right', clip_on=False)
        axins.plot([r2[rep][0], 0.95], [100, 110], color='black', linestyle=':', linewidth=1.5*w, clip_on=False)
        
    elif count == 3:
        axins.text(0.975, 127, clID__label[rep[0]] + '/' + clID__label[rep[1]], ha='right', clip_on=False)
        axins.plot([r2[rep][0], 0.975], [100, 125], color='black', linestyle=':', linewidth=1.5*w, clip_on=False)
        
    elif count == 0:
        axins.text(r2[rep][0], 127, clID__label[rep[0]], ha='left', clip_on=False)
        axins.plot([r2[rep][0], r2[rep][0]], [100, 125], color='black', linestyle=':', linewidth=1.5*w, clip_on=False)

    elif count == 2:
        axins.text(1.005, 112, clID__label[rep[0]], ha='left', clip_on=False)
        axins.plot([r2[rep][0], 1.005], [100, 110], color='black', linestyle=':', linewidth=1.5*w, clip_on=False)
        

axins.set(xlim=[0.9, 1], ylim=[0, 100])
axins.set_xticks([0.9, 0.95, 1]);
axins.set_yticks([0, 50, 100]);


plt.xlabel("Pearson's $r$", fontsize=14, labelpad=8);
plt.ylabel('Number of combinations', fontsize=14, labelpad=8);

#plt.savefig(path_to_save_figs + '2si_pairwise_pearsons_r.pdf', dpi=300, bbox_inches='tight')    


In [ ]:
# calculating p-values

cmap = matplotlib.colors.ListedColormap(['#2e240a', '#69592f', '#a49766', '#d1c7a0', '#e9e0c8'][::-1])  
bounds = [-2.5, -2, -1.5, -1, -0.5, 0]  
norm = matplotlib.colors.BoundaryNorm(bounds, cmap.N)

in_vitro_pvals = {}
for i, j in itertools.combinations(clIDs_ordered_in_vitro, 2):
     
    # normalizing by the fraction at t = 3 days 
    # (not averaging across replicates here, which is different from before)
    t = 3
    norms_i = dict([(count, read_counts_by_cell_line[sample_id][i] / sum(read_counts_by_cell_line[sample_id][1:])) for count, sample_id in enumerate(in_vitro_samples[t])])
    norms_j = dict([(count, read_counts_by_cell_line[sample_id][j] / sum(read_counts_by_cell_line[sample_id][1:])) for count, sample_id in enumerate(in_vitro_samples[t])])

    t = 23 # last time point
    _, p_val = sp.stats.ttest_ind([read_counts_by_cell_line[sample_id][i] / sum(read_counts_by_cell_line[sample_id][1:]) / norms_i[count] for count, sample_id in enumerate(in_vitro_samples[t])], 
                                  [read_counts_by_cell_line[sample_id][j] / sum(read_counts_by_cell_line[sample_id][1:]) / norms_j[count] for count, sample_id in enumerate(in_vitro_samples[t])],
                                  equal_var=False)
    
    in_vitro_pvals[(i, j)] = p_val

pval_matrix = np.full((len(clIDs_ordered_in_vitro), len(clIDs_ordered_in_vitro)), np.nan)  # starting with an empty matrix

clID_to_index = {clID: idx for idx, clID in enumerate(clIDs_ordered_in_vitro)}
for (i, j), p_val in in_vitro_pvals.items():
    idx_i, idx_j = clID_to_index[i], clID_to_index[j]
    pval_matrix[idx_i, idx_j] = np.log10(p_val)
    pval_matrix[idx_j, idx_i] = np.log10(p_val)


plt.figure(figsize=(8, 6.5))
ax = sns.heatmap(pval_matrix, cmap=cmap, norm=norm, vmin=-2.5, vmax=0, linecolor='black', linewidths=0.5,
                 xticklabels=[clID__label[x] + '   ' for x in clIDs_ordered_in_vitro], yticklabels=[clID__label[x] + '   ' for x in clIDs_ordered_in_vitro])

for i in range(len(clIDs_ordered_in_vitro)):
    ax.scatter([-0.7, i + 0.5], [i + 0.5, 33.7], color=clID__color[clIDs_ordered_in_vitro[i]], edgecolor='black', linewidth=w, s=30, clip_on=False)

    
ax.annotate('', xy=(-5, 3 + 0.5), xycoords='data', xytext=(-5, 2 + 0.5), textcoords='data',
             arrowprops=dict(arrowstyle='-', color='0', patchA=None, patchB=None, connectionstyle='bar,fraction=0.7'), 
             annotation_clip=False);

ax.annotate('', xy=(-5, 6 + 0.5), xycoords='data', xytext=(-5, 4 + 0.5), textcoords='data',
             arrowprops=dict(arrowstyle='-', color='0', patchA=None, patchB=None, connectionstyle='bar,fraction=0.4'), 
             annotation_clip=False);

ax.annotate('', xy=(-5, 19 + 0.5), xycoords='data', xytext=(-5, 18 + 0.5), textcoords='data',
             arrowprops=dict(arrowstyle='-', color='0', patchA=None, patchB=None, connectionstyle='bar,fraction=0.7'), 
             annotation_clip=False);

ax.annotate('', xy=(-5, 23 + 0.5), xycoords='data', xytext=(-5, 22 + 0.5), textcoords='data',
             arrowprops=dict(arrowstyle='-', linestyle=':', color='0', patchA=None, patchB=None, connectionstyle='bar,fraction=0.7'), 
             annotation_clip=False, rotation=90);  

ax.annotate('', xy=(3 + 0.5, 38), xycoords='data', xytext=(2 + 0.5, 38), textcoords='data',
             arrowprops=dict(arrowstyle='-', color='0', patchA=None, patchB=None, connectionstyle='bar,fraction=0.7'), 
             annotation_clip=False);

ax.annotate('', xy=(6 + 0.5, 38), xycoords='data', xytext=(4 + 0.5, 38), textcoords='data',
             arrowprops=dict(arrowstyle='-', color='0', patchA=None, patchB=None, connectionstyle='bar,fraction=0.4'), 
             annotation_clip=False);

ax.annotate('', xy=(19 + 0.5, 38), xycoords='data', xytext=(18 + 0.5, 38), textcoords='data',
             arrowprops=dict(arrowstyle='-', color='0', patchA=None, patchB=None, connectionstyle='bar,fraction=0.7'), 
             annotation_clip=False);

ax.annotate('', xy=(23 + 0.5, 38), xycoords='data', xytext=(22 + 0.5, 38), textcoords='data',
             arrowprops=dict(arrowstyle='-', linestyle=':', color='0', patchA=None, patchB=None, connectionstyle='bar,fraction=0.7'), 
             annotation_clip=False, rotation=90);  
    
    
colorbar = ax.collections[0].colorbar
colorbar.set_ticks([-2.5, -2, -1.5, -1, -0.5, 0])  

pos = colorbar.ax.get_position()  
colorbar.ax.set_position([pos.x0 - 0.0125, pos.y0, pos.width - 0.0125, pos.height]) 
ax.tick_params(axis='both', which='both', length=0) 

for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_edgecolor('black') 
    spine.set_linewidth(0.5)   

for b in [-0.5, -1, -1.5, -2]:
    colorbar.ax.axhline(b, color='black', linewidth=w)
colorbar.ax.set_ylabel('') 
colorbar.ax.set_title('$\log_{10}$\np-value', ha='left', position=(-0.1, 0.8), fontsize=12, pad=12) 

colorbar.outline.set_visible(True)  
colorbar.outline.set_edgecolor('black')  
colorbar.outline.set_linewidth(w)  



plt.title('$t=' + str(t) + '$' + ' days ' + '$in\,vitro$');

#plt.savefig(path_to_save_figs + 'fig2_supp_in_vitro_t=' + str(t) + 'days_pvals.pdf', dpi=300, bbox_inches='tight') 

In [ ]:
clID__in_vitro_exp = {}
clID__in_vitro_exp_err = {}
for clID in clIDs_ordered_in_vitro:
     
    t = 3 # normalizing by the fraction at t = 3 days 
    norm = np.mean([read_counts_by_cell_line[sample_id][clID] / sum(read_counts_by_cell_line[sample_id][1:]) for sample_id in in_vitro_samples[t]])

    t = 23 # last time point
    y = [read_counts_by_cell_line[sample_id][clID] / sum(read_counts_by_cell_line[sample_id][1:]) / norm for sample_id in in_vitro_samples[t]]
    
    clID__in_vitro_exp[clID] = np.mean(y)
    clID__in_vitro_exp_err[clID] = [np.std(y), np.std(y)]
    
    
ax = utils.metric_bar_plot(clIDs_ordered_in_vitro, clID__in_vitro_exp, clID__in_vitro_exp_err, 
                           ytitle='Relative ' + '$\it{in\,vitro}$' + ' expansion')

ax.annotate('', xy=(2, 0.0015), xycoords='data', xytext=(3, 0.0015), textcoords='data',
            arrowprops=dict(arrowstyle='-', color='0', patchA=None, patchB=None, connectionstyle='bar,angle=180,fraction=-0.4'), 
            annotation_clip=False);

ax.annotate('', xy=(4, 0.0015), xycoords='data', xytext=(6, 0.0015), textcoords='data',
            arrowprops=dict(arrowstyle='-', color='0', patchA=None, patchB=None, connectionstyle='bar,angle=180,fraction=-0.2'), 
            annotation_clip=False);

ax.annotate('', xy=(18, 0.0015), xycoords='data', xytext=(19, 0.0015), textcoords='data',
            arrowprops=dict(arrowstyle='-', color='0', patchA=None, patchB=None, connectionstyle='bar,angle=180,fraction=-0.4'), 
            annotation_clip=False);

ax.annotate('', xy=(22, 0.0015), xycoords='data', xytext=(23, 0.0015), textcoords='data',
            arrowprops=dict(arrowstyle='-', linestyle=':', color='0', patchA=None, patchB=None, connectionstyle='bar,angle=180,fraction=-0.4'), 
            annotation_clip=False);

#plt.savefig(path_to_save_figs + 'fig2_supp_in_vitro_t=8days_exp_barplot.pdf', dpi=300, bbox_inches='tight') 


In [ ]:
plt.figure(figsize=(8, 4));
plt.gca().spines[['top', 'right']].set_visible(False)
plt.gca().spines['bottom'].set_position(('axes', -0.025))
plt.gca().spines['left'].set_position(('axes', -0.025))

def doubling_time(N1, N2, f1, f2, dt): # in days
    r = (f2 * N2) / (f1 * N1)
    return dt * math.log(2) / math.log(r)

ys = []; errs = []
for clID in clIDs_ordered_in_vitro:
    ts = []
    t_minus_1 = 3 # starting from time point 1, not 0
    samples_minus_1 = in_vitro_samples[t_minus_1]
    for t, samples in in_vitro_samples.items():
        if t == 3:
            continue
        dt = t - t_minus_1
        for rep in range(3):
            ts.append(doubling_time(metadata.loc[samples[rep]]['initial number of cells'], 
                                    metadata.loc[samples[rep]]['lung weight, g (in vivo) / cells (in vitro)'],
                                    read_counts_by_cell_line[samples_minus_1[rep]][clID] / sum(read_counts_by_cell_line[samples_minus_1[rep]][1:]),
                                    read_counts_by_cell_line[samples[rep]][clID] / sum(read_counts_by_cell_line[samples[rep]][1:]), 
                                    dt))
        t_minus_1 = t
        samples_minus_1 = samples

    ys.append(np.mean(ts) - 0.5); errs.append(np.std(ts))
    


cs = [colors.to_rgba(i) for i in range(len(clIDs_ordered_in_vitro))]
    
plt.bar(range(len(clIDs_ordered_in_vitro)), ys, width=0.8, bottom=0.5, color=[clID__color[i] for i in clIDs_ordered_in_vitro], edgecolor='black', linewidth=0.5, clip_on=False)
plt.errorbar(range(len(clIDs_ordered_in_vitro)), [y + 0.5 for y in ys], yerr=errs, color='black', linewidth=0.75, alpha=1, zorder=1, ls='none', clip_on=False)

plt.yticks([0, 0.25, 0.5, 0.75, 1, 1.25, 1.5], ['0', '0.25', '0.5', '0.75', '1', '1.25', '1.5'], size=12)
plt.ylim(0.5, 1.5)

plt.xlim(0, count);
plt.xticks(range(len(clIDs_ordered_in_vitro)), [clID__label[x].replace('_', ' ') for x in clIDs_ordered_in_vitro], size=12, rotation=90);

plt.ylabel('Doubling time ' + '$\it{in\,vitro}$' + ', days', fontsize=14, labelpad=14);

plt.annotate('', xy=(2, 0.22), xycoords='data', xytext=(3, 0.22), textcoords='data',
             arrowprops=dict(arrowstyle='-', color='0', patchA=None, patchB=None, connectionstyle='bar,angle=180,fraction=-0.4'), 
             annotation_clip=False);

plt.annotate('', xy=(4, 0.22), xycoords='data', xytext=(6, 0.22), textcoords='data',
             arrowprops=dict(arrowstyle='-', color='0', patchA=None, patchB=None, connectionstyle='bar,angle=180,fraction=-0.2'), 
             annotation_clip=False);

plt.annotate('', xy=(18, 0.22), xycoords='data', xytext=(19, 0.22), textcoords='data',
             arrowprops=dict(arrowstyle='-', color='0', patchA=None, patchB=None, connectionstyle='bar,angle=180,fraction=-0.4'), 
             annotation_clip=False);

plt.annotate('', xy=(22, 0.22), xycoords='data', xytext=(23, 0.22), textcoords='data',
             arrowprops=dict(arrowstyle='-', linestyle=':', color='0', patchA=None, patchB=None, connectionstyle='bar,angle=180,fraction=-0.4'), 
             annotation_clip=False);

#plt.savefig(path_to_save_figs + 'fig2_supp_in_vitro_doubling_times.pdf', dpi=300, bbox_inches='tight')  


### In vivo plots 

In [ ]:
# making a list of 3w F1 iv lung samples from exp. one
what = ['1', 20, 'C57B6/129S F1', 'intravenous', 'lung']
samples = list(set(metadata.loc[(metadata['experiment'] == what[0]) & 
                                (metadata['time point, d'] > what[1]) & # be careful here
                                (metadata['genotype'] == what[2]) &
                                (metadata['injection method'] == what[3]) & 
                                (metadata['tissue'] == what[4])
].apply(lambda row: row['notes'].split()[0] if row['notes'] != 'none' else row.name, axis=1).tolist())) # choosing the correct name for merged counts



In [ ]:
clID__in_vivo_exp, clID__in_vivo_exp_avg_relative_to_mean, clID__in_vivo_exp_err, clIDs_sorted = utils.bootstrap_relative_burden(samples, read_counts=read_counts_by_cell_line, cell_counts=cell_counts_by_cell_line, metadata=metadata)


In [ ]:
ax = utils.metric_bar_plot(clIDs_sorted[:-1], clID__in_vivo_exp_avg_relative_to_mean, clID__in_vivo_exp_err, 
                           ytitle='Relative lung metastatic burden\n(F1 mice at 3 weeks)', 
                           ylims=[1e-4, 1e2], yticks=[1e-4, 1e-3, 1e-2, 1e-1, 1e0, 1e1, 1e2], ylabels=['0.0001', '0.001', '0.01', '0.1', '1', '10', '100'])

ax.plot([-1, len(clIDs_sorted[:-1]) + 0.25], [1, 1], ':', clip_on=False, linewidth=2*w, color='black', zorder=-1)
ax.plot([len(clIDs_sorted[:-1]) + 0.25, len(clIDs_sorted[:-1]) + 0.25], [1, 4], ':', clip_on=False, linewidth=2*w, color='black', zorder=-1)
ax.plot([len(clIDs_sorted[:-1]), len(clIDs_sorted[:-1]) + 0.25], [4, 4], ':', clip_on=False, linewidth=2*w, color='black', zorder=-1)

ax.text(len(clIDs_sorted[:-1]) - 0.25, 4, 'average across\nall cell lines\n(including Panc02)', va='center', ha='right')


ax.annotate('', xy=(2, 0.000002), xycoords='data', xytext=(4, 0.000002), textcoords='data',
            arrowprops=dict(arrowstyle='-', color='0', patchA=None, patchB=None, connectionstyle='bar,angle=180,fraction=-0.2'), 
            annotation_clip=False);

ax.annotate('', xy=(28, 0.000002), xycoords='data', xytext=(31, 0.000002), textcoords='data',
            arrowprops=dict(arrowstyle='-', color='0', patchA=None, patchB=None, connectionstyle='bar,angle=180,fraction=-0.2'), 
            annotation_clip=False);

ax.annotate('', xy=(29, 0.000002), xycoords='data', xytext=(30, 0.000002), textcoords='data',
            arrowprops=dict(arrowstyle='-', color='0', patchA=None, patchB=None, connectionstyle='bar,angle=180,fraction=-0.4'), 
            annotation_clip=False);

ax.annotate('', xy=(17, 0.000002), xycoords='data', xytext=(20, 0.000002), textcoords='data',
            arrowprops=dict(arrowstyle='-', linestyle=':', color='0', patchA=None, patchB=None, connectionstyle='bar,angle=180,fraction=-0.2'), 
            annotation_clip=False);

#plt.savefig(path_to_save_figs + 'fig2_relative_in_vivo_exp.pdf', dpi=300, bbox_inches='tight')  


In [ ]:
ax = utils.comparison_scatter_plot([x for x in clIDs if x not in ['GATC', 'AGCA']],
                                   clID__x=clID__in_vivo_exp_avg_relative_to_mean, clID__xerr=clID__in_vivo_exp_err,
                                   clID__y=clID__in_vitro_exp, clID__yerr=clID__in_vitro_exp_err,
                                   xtitle='Relative lung metastatic burden\n(F1 mice at 3 weeks)', ytitle='Relative ' + '$\it{in\,vitro}$' + ' expansion',
                                   lims=[1e-3, 1e2], ticks=[1e-3, 1e-2, 1e-1, 1e0, 1e1, 1e2], labels=['0.001', '0.01', '0.1', '1', '10', '100'])

#plt.savefig(path_to_save_figs + 'fig2_relative_in_vivo_exp_vs_relative_in_vitro_expansion.pdf', dpi=300, bbox_inches='tight')  

In [ ]:
# now just want to plot mouse data from pooled experiments

what = [20, 'intravenous', 'lung'] # from both experiments, both sets of mice
samples_weight = list(set(metadata.loc[(metadata['time point, d'] > what[0]) & # be careful here
                         (metadata['injection method'] == what[1]) & 
                         (metadata['tissue'] == what[2])
].apply(lambda row: (row['notes'].split()[0] if row['notes'] != 'none' else row.name, 
                     row['lung weight, g (in vivo) / cells (in vitro)'],
                     row['experiment'],
                     row['genotype']), axis=1).tolist()))


plt.figure(figsize=(6, 6))

used_labels = set(); xs = []; ys = []
for (sample, weight, exp, gen) in samples_weight:
    
    if gen == 'C57B6/129S F1':
        if exp == '1':
            color = '#898989'
            label = 'C57B6/129S F1'
        elif exp == '2':
            color = '#f37c79'
            label = 'C57B6/129S F1, transplanted after passaging'
            
    elif gen == 'Rag1-/-':
        if exp == '1':
            color = 'black'
            label = 'Rag1-/-'
        elif exp == '2':
            color = '#941c52'
            label = 'Rag1-/-, transplanted after passaging'
        
    x = sum(cell_counts_by_cell_line[sample][1:])
    xs.append(x); ys.append(weight)
    
    if label not in used_labels:
        plt.scatter(x, weight, s=80, color=color, linewidth=0, label=label)
        used_labels.add(label) 
    else:
        plt.scatter(x, weight, s=80, color=color, linewidth=0)
        
spearmans_rho = sp.stats.spearmanr(xs, ys) # rank correlation coefficient
pearsons_r = sp.stats.pearsonr(xs, ys) # correlation coefficient
plt.text(0.7e5, 0.85, 'ρ = ' + '{0:.2f}'.format(spearmans_rho[0]) + ', p = ' + '{0:.0e}'.format(spearmans_rho[1]), size=12)
plt.text(0.7e5, 0.8, 'r = ' + '{0:.2f}'.format(pearsons_r[0]) + ', p = ' + '{0:.0e}'.format(pearsons_r[1]), size=12)

plt.axhline(0.2, color='#d9d9d9', zorder=-100)
plt.xscale('log')
plt.minorticks_off()

plt.xlim(5e4, 2e8)
plt.ylim(0, 1.2)

plt.xticks(size=12)
plt.yticks(size=12)

handles, labels = plt.gca().get_legend_handles_labels()
order = [0, 2, 1, 3]
plt.legend([handles[idx] for idx in order],[labels[idx] for idx in order], frameon=False)

plt.text(1.75e8, 0.225, 'normal lung', size=12, color='#898989', ha='right')

plt.ylabel('Lung weight (g)', fontsize=14, labelpad=6);
plt.xlabel('Lung metastatic burden at 3 weeks, cells', fontsize=14, labelpad=6);

#plt.savefig(path_to_save_figs + 'fig2_si_lung_weight_vs_total_burden.pdf', dpi=300, bbox_inches='tight')  

In [ ]:
# plotting lung weight data vs relative expansion;
# need to retreive lung weights for mice from figure 1 first:
lung_weights = pd.read_csv('../../Data/tissue_weight.csv').drop(['Eartag', 'Genotype', 'Injected on', 'Collected on', 'Comments'], axis=1)
lung_weights_avg = lung_weights.groupby('Injected (vol. 200 ul)').mean().drop(['Collected (wks)', 'Pancreas wt.', 'Spleen wt.', 'Liver wt.'], axis=1).to_dict()['Lung wt. ']
lung_weights_stds = lung_weights.groupby('Injected (vol. 200 ul)').std().drop(['Collected (wks)', 'Pancreas wt.', 'Spleen wt.', 'Liver wt.'], axis=1).to_dict()['Lung wt. ']


plt.figure(figsize=(6, 6))

xs = []; ys = []
processed_clIDs = []
for clID in [x for x in clIDs if x not in ['GATC', 'AGCA']]:
    cell_line = clID__label[clID]
        
    x = clID__in_vivo_exp_avg_relative_to_mean[clID]
    x_err = clID__in_vivo_exp_err[clID]
    
    # including cell lines that were transplanted at a lower dose for now:
    if cell_line in ['6694c2', '6419c5', 'BF1987', '0688M', 'FC1245', 'FC1245', 'KPC-JH', '0755A', '2838c3']:
        cell_line = cell_line + '*'
        y = lung_weights_avg[cell_line]
        y_err = lung_weights_stds[cell_line]
        processed_clIDs.append(clID)

    else:
        if cell_line in lung_weights_avg:
            y = lung_weights_avg[cell_line]
            y_err = lung_weights_stds[cell_line]
            processed_clIDs.append(clID)
        
        else:
            cell_line = 'UN-'+ '-'.join(re.split('(\d+)', cell_line))[:-1]
            y = lung_weights_avg[cell_line]
            y_err = lung_weights_stds[cell_line]
            processed_clIDs.append(clID)

    xs.append(x); ys.append(y)
    
    plt.scatter(x, y, color=clID__color[clID], s=100, edgecolor='black', linewidth=0.5)
    plt.errorbar([x], y, xerr=[[x_err[0]], [x_err[1]]], yerr=y_err, color='black', linewidth=0.5, zorder=-1)
    
    
spearmans_rho = sp.stats.spearmanr(xs, ys) # rank correlation coefficient
pearsons_r = sp.stats.pearsonr(xs, ys) # correlation coefficient
plt.text(3e-3, 1.12, 'ρ = ' + '{0:.2f}'.format(spearmans_rho[0]) + ', p = ' + '{0:.0e}'.format(spearmans_rho[1]), size=12)
plt.text(3e-3, 1.07, 'r = ' + '{0:.2f}'.format(pearsons_r[0]) + ', p = ' + '{0:.0e}'.format(pearsons_r[1]), size=12)
    
plt.axhline(0.2, color='#d9d9d9', zorder=-100)
plt.xscale('log')
plt.minorticks_off()

plt.xlim(0.2e-2, 5e1)
plt.ylim(0, 1.2)

plt.xticks(size=12)
plt.yticks(size=12)

plt.ylabel('Lung weight (g)', fontsize=14, labelpad=6);
plt.xlabel('Relative lung metastatic burden\n (F1 mice at 3 weeks)', fontsize=14, labelpad=6);

#plt.savefig(path_to_save_figs + 'fig2_si_lung_weight_vs_relative_in_vivo_exp.pdf', dpi=300, bbox_inches='tight')  

In [ ]:
# histology data:
histology = pd.read_csv('../../Data/histology_tumor_area.csv', index_col='cell line')

plt.figure(figsize=(6, 6))

xs = []; ys = []
for clID in [x for x in clIDs if x not in ['GATC', 'AGCA']]:
    cell_line = clID__label[clID]
    
    if cell_line not in histology.index:
        continue
        
    x = clID__in_vivo_exp_avg_relative_to_mean[clID]
    x_err = clID__in_vivo_exp_err[clID]
    y = histology.loc[cell_line]['percentage tumor area']
    m = histology.loc[cell_line]['set']
    
    if m == 1:
        plt.scatter(x, y, color=clID__color[clID], marker='^', s=100, edgecolor='black', linewidth=0.5)
    else:
       # plt.scatter(x, y, color='black', marker=matplotlib.markers.MarkerStyle('s', fillstyle='bottom'), s=100, edgecolor='black', linewidth=0.5)
        plt.scatter(x, y, color=clID__color[clID], marker='s', s=100, edgecolor='black', linewidth=0.5)
    
    xs.append(x); ys.append(y)
    
    
    plt.errorbar([x], y, xerr=[[x_err[0]], [x_err[1]]], color='black', linewidth=0.5, zorder=-1)
    
    
spearmans_rho = sp.stats.spearmanr(xs, ys) # rank correlation coefficient
pearsons_r = sp.stats.pearsonr(xs, ys) # correlation coefficient
plt.text(3e-3, 92, 'ρ = ' + '{0:.2f}'.format(spearmans_rho[0]) + ', p = ' + '{0:.0e}'.format(spearmans_rho[1]), size=12)
plt.text(3e-3, 87, 'r = ' + '{0:.2f}'.format(pearsons_r[0]) + ', p = ' + '{0:.0e}'.format(pearsons_r[1]), size=12)
    
plt.xscale('log')
plt.minorticks_off()

plt.xlim(0.2e-2, 5e1)
plt.ylim(0, 100)

plt.xticks(size=12)
plt.yticks(size=12)

plt.ylabel('Percentage tumor area', fontsize=14, labelpad=6);
plt.xlabel('Relative lung metastatic burden\n (F1 mice at 3 weeks)', fontsize=14, labelpad=6);

#plt.savefig(path_to_save_figs + 'fig2_si_histology_vs_relative_in_vivo_exp.pdf', dpi=300, bbox_inches='tight')  

### Supplementary figure: read to cell number conversion

In [ ]:
fig, ax = plt.subplots(figsize=(2, 4))
ax.spines[['top', 'right', 'bottom']].set_visible(False)

processed_samples = []; used_labels = []
np.random.seed(0) 
for sample_id in metadata.index:
    if sample_id in processed_samples:
        continue
    if metadata.loc[sample_id]['spike-in added'] == False:
        continue

    if metadata.loc[sample_id]['injection method'] == 'intravenous':
        c = '#b9b9b9'; z=-1
    else:
        c = 'black'; z=0
        
    if metadata.loc[sample_id]['notes'] == 'none': # no need to merge anything
        plt.scatter(1 + np.random.normal(0, 0.012), read_counts_by_cell_line.loc['GATC'][sample_id] / 50000, color=c, zorder=z)    
    elif 'one' in metadata.loc[sample_id]['notes']:
        plt.scatter(1 + np.random.normal(0, 0.012), sum(read_counts_by_cell_line[subsample][0] for subsample in metadata.loc[sample_id]['notes'].split()[0].split(':')) / 50000, color=c, zorder=z)
        processed_samples.extend([subsample for subsample in metadata.loc[sample_id]['notes'].split()[0].split(':')])
    else: 
        plt.scatter(1 + np.random.normal(0, 0.012), sum(read_counts_by_cell_line[subsample][0] / 50000 for subsample in metadata.loc[sample_id]['notes'].split()[0].split(':')), color=c, zorder=z)
        processed_samples.extend([subsample for subsample in metadata.loc[sample_id]['notes'].split()[0].split(':')])
    
# for legend
plt.scatter(-1, 1, color='#b9b9b9', label = 'intravenous')
plt.scatter(-1, 1, color='black', label = 'other')
    
plt.yscale('log')
plt.minorticks_off()
plt.xlim(0.95, 1.05)
plt.ylim(1e-4, 1e1)
plt.xticks([], []);
plt.yticks(size=12)

plt.legend(frameon=False, handletextpad=0.1, labelspacing=0.1, bbox_to_anchor=(0.8, 0.16))

plt.ylabel('Reads per cell', fontsize=14, labelpad=6);
plt.xlabel('Sample', fontsize=14, labelpad=6);

#plt.savefig(path_to_save_figs + 'fig2_si_reads_per_cell.pdf', dpi=300, bbox_inches='tight')  

### Supplementary fugure: number of unique barcodes recovered from the first experiment

In [ ]:
# making a list of 3w F1 iv lung samples from exp. one
what = ['1', 20, 'C57B6/129S F1', 'intravenous', 'lung']
samples = list(set(metadata.loc[(metadata['experiment'] == what[0]) & 
                                (metadata['time point, d'] > what[1]) & # be careful here
                                (metadata['genotype'] == what[2]) &
                                (metadata['injection method'] == what[3]) & 
                                (metadata['tissue'] == what[4])
].apply(lambda row: row['notes'].split()[0] if row['notes'] != 'none' else row.name, axis=1).tolist())) # choosing the correct name for merged counts


In [ ]:
# getting tumor sizes:
sample_clID_barcode__count = utils.convert_barcode_reads_to_cell_counts(samples, path_to_data=path_to_reads, read_counts=read_counts_by_cell_line, metadata=metadata)

In [ ]:
plt.figure(figsize=(8, 3))
plt.gca().spines[['top', 'right', 'bottom']].set_visible(False)

clID__n_tumors = {}
for sample in samples:
    for i, clID in enumerate(clIDs[1:]):
        if clID not in clID__n_tumors:
            clID__n_tumors[clID] = []
        clID__n_tumors[clID].append(len([x for x in sample_clID_barcode__count[sample][clID].values() if x >= 100]))
        #plt.scatter(i, n_tumors, color='#00000000', edgecolor='black', linewidth=0.8)

for i, clID in enumerate(clIDs_sorted):
    plt.scatter(i + np.random.normal(0, 0.1, len(clID__n_tumors[clID])), [0.2 if x == 0 else x for x in clID__n_tumors[clID]], 
                color=clID__color[clID], edgecolor='black', linewidth=0.5, clip_on=False)

plt.yscale('log')
plt.minorticks_off()

plt.ylim(2e-1, 1e3)
plt.xlim(-1, i + 1)

# make sure that the order is correct
plt.xticks(range(len(clIDs_sorted)), []);
plt.gca().set_xticklabels([clID__label[x] + '  ' for x in clIDs_sorted], size=10, rotation=90)
plt.gca().xaxis.set_tick_params(length=0, width=0, which='both')
plt.yticks([2e-1, 1e0, 1e1, 1e2, 1e3], ['0', '$10^{0}$', '$10^{1}$', '$10^{2}$', '$10^{3}$'], size=12)


for i in np.logspace(np.log10(0.35), np.log10(0.4), 10):
    plt.text(-1.4, i, '|', rotation=-40, size=12, color='white')
plt.text(-1.4, 0.35, '|', rotation=-40, size=12)
plt.text(-1.4, 0.4, '|', rotation=-40, size=12)

plt.annotate('', xy=(2, 0.015), xycoords='data', xytext=(4, 0.015), textcoords='data',
             arrowprops=dict(arrowstyle='-', color='0', patchA=None, patchB=None, connectionstyle='bar,angle=180,fraction=-0.2'), 
             annotation_clip=False);

plt.annotate('', xy=(28, 0.015), xycoords='data', xytext=(31, 0.015), textcoords='data',
             arrowprops=dict(arrowstyle='-', color='0', patchA=None, patchB=None, connectionstyle='bar,angle=180,fraction=-0.2'), 
             annotation_clip=False);

plt.annotate('', xy=(29, 0.015), xycoords='data', xytext=(30, 0.015), textcoords='data',
             arrowprops=dict(arrowstyle='-', color='0', patchA=None, patchB=None, connectionstyle='bar,angle=180,fraction=-0.4'), 
             annotation_clip=False);

plt.annotate('', xy=(17, 0.015), xycoords='data', xytext=(20, 0.015), textcoords='data',
             arrowprops=dict(arrowstyle='-', linestyle=':', color='0', patchA=None, patchB=None, connectionstyle='bar,angle=180,fraction=-0.2'), 
             annotation_clip=False);

plt.ylabel('Number of unique barcodes\n(from metastasis ≥ 100 cells)', fontsize=13, labelpad=6);

#plt.savefig(path_to_save_figs + 'fig2_si_n_barcodes.pdf', dpi=300, bbox_inches='tight')  

In [ ]:
# to do:
# 1. edit metadata to change the path to data, or better reorganize folders? -- do this today -- kind of done, but honestly need to just remake all files...
# 2. figure out how to run stuff on the cluster + run it; or else check what has been processed already??
# 3. I think that all of the figures that Monte wanted are somewhat straightforward; 
# the plan is to try to make them tomorrow - Fri and leave everything else on the weekend
